# Naive Bayes on complaint narratives

**CSCI 5612 — Module 3, Project Part 3**

When an owner files a safety complaint they describe what happened in their own
words. Can those words alone tell us a crash occurred — without knowing the
make, the model, or anything else about the vehicle?

Multinomial NB expects non-negative integer features, which is exactly what a
word-count matrix is. Bernoulli NB is fitted on the same split for comparison.

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB, MultinomialNB

pd.set_option("display.width", 200)
RANDOM_STATE, TEST_SIZE = 5612, 0.25

## 1. Labelled data

Supervised learning needs a known answer on every row. The label here is
recorded fact: the federal complaint form has a field for whether a crash
occurred.

In [2]:
c = pd.read_csv("../data/clean/complaints_clean.csv.gz", low_memory=False)
c["summary"] = c["summary"].fillna("").astype(str)
c = c[c["summary"].str.split().str.len() >= 5]          # too short to model
c["label"] = np.where(c["crash"], "Crash", "No crash")

print("labelled complaints:", f"{len(c):,}")
print(c["label"].value_counts(normalize=True).round(4))
c[["make", "model", "summary", "label"]].head(3)

labelled complaints: 83,282
label
No crash    0.9526
Crash       0.0474
Name: proportion, dtype: float64


,make,model,summary,label
0,honda,accord,DRIVER SIDE VISOR DROPS DOWN INTO MY FIELD OF ...,No crash
1,honda,accord,A very loud static noise occurs about 5 minute...,No crash
2,honda,accord,The radio outputs very loud static whether the...,No crash


## 2. A disjoint train / test split

The two sets must share no rows. A model tested on what it trained on is
recalling, not generalising. The split is **stratified** so both sets carry the
same 4.7% crash rate.

In [3]:
train, test = train_test_split(c, test_size=TEST_SIZE,
                               random_state=RANDOM_STATE, stratify=c["label"])
overlap = set(train["odiNumber"]) & set(test["odiNumber"])
print(f"train {len(train):,}   test {len(test):,}")
print("overlapping complaint IDs:", len(overlap))
assert not overlap, "sets are not disjoint"

train 62,461   test 20,821
overlapping complaint IDs: 0


## 3. Vectorise

The vectoriser is fitted on the **training set only**. Fitting it on everything
first would leak test vocabulary into the model.

In [4]:
vec = CountVectorizer(stop_words="english", min_df=5, max_features=6000)
Xtr = vec.fit_transform(train["summary"])     # fit + transform on TRAIN
Xte = vec.transform(test["summary"])          # transform ONLY on TEST
ytr, yte = train["label"].values, test["label"].values

print("vocabulary:", f"{len(vec.vocabulary_):,} terms")
print("train matrix", Xtr.shape, "| min value", Xtr.min(), "| dtype", Xtr.dtype)

vocabulary: 6,000 terms
train matrix (62461, 6000) | min value 0 | dtype int64


## 4. Fit both variants, and a baseline that ignores the text entirely

In [5]:
mnb = MultinomialNB(alpha=1.0).fit(Xtr, ytr)
bnb = BernoulliNB(alpha=1.0, binarize=0.5).fit(Xtr, ytr)
pm, pb = mnb.predict(Xte), bnb.predict(Xte)
pmaj = np.full(len(yte), pd.Series(ytr).value_counts().idxmax())

rows = []
for name, pred in (("Multinomial NB", pm), ("Bernoulli NB", pb),
                   ("Majority-class baseline", pmaj)):
    rows.append({"model": name,
                 "accuracy": accuracy_score(yte, pred),
                 "f1_crash": f1_score(yte, pred, pos_label="Crash", zero_division=0)})
pd.DataFrame(rows).round(4)

,model,accuracy,f1_crash
0,Multinomial NB,0.9441,0.5848
1,Bernoulli NB,0.9517,0.6125
2,Majority-class baseline,0.9526,0.0000


**Read that baseline carefully.** Answering "no crash" every time scores higher
accuracy than Multinomial NB while finding zero crashes. With a 4.7% positive
class, accuracy on its own is not a meaningful measure.

In [6]:
print(classification_report(yte, pm, zero_division=0))
pd.DataFrame(confusion_matrix(yte, pm, labels=["No crash", "Crash"]),
             index=["actual No crash", "actual Crash"],
             columns=["pred No crash", "pred Crash"])

              precision    recall  f1-score   support

       Crash       0.45      0.83      0.58       987
    No crash       0.99      0.95      0.97     19834

    accuracy                           0.94     20821
   macro avg       0.72      0.89      0.78     20821
weighted avg       0.97      0.94      0.95     20821



,pred No crash,pred Crash
actual No crash,18839,995
actual Crash,168,819


## 5. Why smoothing is required

If a word was never seen in a class during training its probability is zero, and
because the scores are products, one zero collapses the whole class score. Adding
alpha to every count keeps every probability strictly positive.

In [7]:
rows = []
for a in [1e-10, 1e-4, 0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]:
    p = MultinomialNB(alpha=a).fit(Xtr, ytr).predict(Xte)
    rows.append({"alpha": a, "accuracy": accuracy_score(yte, p),
                 "f1_crash": f1_score(yte, p, pos_label="Crash", zero_division=0)})
pd.DataFrame(rows).round(4)

,alpha,accuracy,f1_crash
0,0.0000,0.9508,0.5724
1,0.0001,0.9521,0.6051
2,0.0100,0.9500,0.6013
3,0.1000,0.9478,0.5958
4,0.5000,0.9451,0.5866
5,1.0000,0.9441,0.5848
6,2.0000,0.9457,0.5912
7,5.0000,0.9524,0.6140
8,10.0000,0.9629,0.6523


## 6. What the model learned

In [8]:
terms = np.array(vec.get_feature_names_out())
cls = list(mnb.classes_)
llr = mnb.feature_log_prob_[cls.index("Crash")] - mnb.feature_log_prob_[cls.index("No crash")]
print("most 'Crash'    :", ", ".join(terms[np.argsort(llr)[-12:]][::-1]))
print("most 'No crash' :", ", ".join(terms[np.argsort(llr)[:12]]))

most 'Crash'    : pole, embankment, guardrail, crashed, concussion, fence, barrier, collided, deploy, bruises, ribs, deer
most 'No crash' : paint, seal, noticeable, peeling, wear, cylinder, leaking, valve, gallons, consumption, trim, compressor


## 7. What this says about the topic

Owners describing a crash write differently from owners describing a fault, and
the giveaway words are physical nouns — the things vehicles hit. Poles,
embankments, guardrails, fences. Non-crash complaints speak the language of
maintenance: peeling paint, leaking seals, oil consumption. Nobody designed
these two vocabularies; they fell out of counting words in eighty thousand
accounts.

The free-text field therefore carries far more information than it is currently
used for. A method simple enough to explain on one page reads all of them at
once and flags more than four in five crash-related reports.

Full discussion: <https://rajeshwar987.github.io/csci5612-project/naivebayes.html>